In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
# ---------------------------------------------------
# 📦 1. Preparar dados
# Estrutura sugerida:
# dataset/
# ├── images/         # imagens ECG com falhas
# └── masks/          # máscaras binárias (linha = 255, fundo = 0)

image_dir = 'dataset/images'
mask_dir = 'dataset/masks'

# Função para carregar imagens e máscaras
def load_data(image_dir, mask_dir, img_size=(256,256)):
    images = []
    masks = []
    files = os.listdir(image_dir)
    for file in files:
        img = cv2.imread(os.path.join(image_dir, file))
        img = cv2.resize(img, img_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = img / 255.0
        images.append(img[..., np.newaxis])

        mask = cv2.imread(os.path.join(mask_dir, file), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, img_size)
        mask = (mask > 127).astype(np.float32)  # binário
        masks.append(mask[..., np.newaxis])
    return np.array(images), np.array(masks)

X, y = load_data(image_dir, mask_dir)
print("✅ Dados carregados:", X.shape, y.shape)

# ---------------------------------------------------
# 🧠 2. Construir U-Net
def build_unet(input_shape):
    inputs = Input(shape=input_shape)

    c1 = Conv2D(16, (3,3), activation='relu', padding='same')(inputs)
    c1 = Conv2D(16, (3,3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2,2))(c1)

    c2 = Conv2D(32, (3,3), activation='relu', padding='same')(p1)
    c2 = Conv2D(32, (3,3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2,2))(c2)

    c3 = Conv2D(64, (3,3), activation='relu', padding='same')(p2)
    c3 = Conv2D(64, (3,3), activation='relu', padding='same')(c3)

    u1 = Conv2DTranspose(32, (2,2), strides=(2,2), padding='same')(c3)
    u1 = concatenate([u1, c2])
    c4 = Conv2D(32, (3,3), activation='relu', padding='same')(u1)
    c4 = Conv2D(32, (3,3), activation='relu', padding='same')(c4)

    u2 = Conv2DTranspose(16, (2,2), strides=(2,2), padding='same')(c4)
    u2 = concatenate([u2, c1])
    c5 = Conv2D(16, (3,3), activation='relu', padding='same')(u2)
    c5 = Conv2D(16, (3,3), activation='relu', padding='same')(c5)

    outputs = Conv2D(1, (1,1), activation='sigmoid')(c5)

    model = Model(inputs, outputs)
    return model

model = build_unet((256,256,1))
model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# ---------------------------------------------------
# 🏋️ 3. Treinar U-Net
checkpoint = ModelCheckpoint('best_unet.h5', save_best_only=True, monitor='val_loss', mode='min')

history = model.fit(
    X, y,
    validation_split=0.2,
    epochs=30,
    batch_size=4,
    callbacks=[checkpoint]
)

# ---------------------------------------------------
# 📈 4. Avaliar e plotar histórico
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend(); plt.title('Training History')
plt.show()

# ---------------------------------------------------
# 🧪 5. Fazer predição em nova imagem (linha azul ou preta)
test_img = cv2.imread('ecg_teste.png')
test_img_resized = cv2.resize(test_img, (256,256))
gray = cv2.cvtColor(test_img_resized, cv2.COLOR_BGR2GRAY) / 255.0
input_tensor = np.expand_dims(gray, axis=(0,-1))

# Carregar melhor modelo
test_model = build_unet((256,256,1))
test_model.load_weights('best_unet.h5')

pred = test_model.predict(input_tensor)[0,...,0]
mask_pred = (pred > 0.5).astype(np.uint8) * 255

# ---------------------------------------------------
# 🪄 6. Pós-processamento para preencher falhas
kernel = np.ones((3,3), np.uint8)
filled = cv2.morphologyEx(mask_pred, cv2.MORPH_CLOSE, kernel)

# Aplicar sobre canal verde (exemplo) para ver resultado
output = test_img_resized.copy()
output[filled==255] = [0,0,0]  # linha preta, ou ajuste cor se for azul

# ---------------------------------------------------
# 📊 Visualizar resultado
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.title('Original'); plt.imshow(test_img_resized)
plt.subplot(1,3,2); plt.title('Mask'); plt.imshow(filled, cmap='gray')
plt.subplot(1,3,3); plt.title('Linha preenchida'); plt.imshow(output)
plt.show()

# ---------------------------------------------------
# 💾 Salvar
cv2.imwrite('ecg_corrigido.png', output)

print("✅ Processamento concluído! Resultado salvo como 'ecg_corrigido.png'")

# ---------------------------------------------------
# ✏️ Instruções resumo:
# 1. Coloque suas imagens em dataset/images e máscaras em dataset/masks
# 2. Ajuste cores e pós-processamento conforme linha preta ou azul
# 3. Treine e aplique em novas imagens ECG
